In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

from ecg.data import (
    get_superclasses,
    load_metadata,
    load_or_cache,
    load_statements,
    split_by_fold,
)
from ecg.features import extract_features

In [2]:
df = load_metadata("../data/raw/ptbxl_database.csv")
statements = load_statements("../data/raw/scp_statements.csv")
df["superclasses"] = df["scp_codes"].apply(
    lambda codes: get_superclasses(codes, statements)
)
len(df)

21799

In [3]:
len_classes = df["superclasses"].apply(len)
len_classes

ecg_id
1        1
2        1
3        1
4        1
5        1
        ..
21833    1
21834    1
21835    1
21836    1
21837    1
Name: superclasses, Length: 21799, dtype: int64

In [4]:
df = df[len_classes > 0]
len(df)

21388

In [5]:
train_df, val_df, test_df = split_by_fold(df)

In [6]:
sorted(set(df["superclasses"].explode()))

['CD', 'HYP', 'MI', 'NORM', 'STTC']

In [7]:
mlb = MultiLabelBinarizer(classes=["CD", "HYP", "MI", "NORM", "STTC"])

y_train = mlb.fit_transform(train_df["superclasses"])
y_val = mlb.transform(val_df["superclasses"])
y_test = mlb.transform(test_df["superclasses"])

In [8]:
X_train = load_or_cache(Path("../data/processed/X_train.npy"), train_df, "../data/raw/")
X_val = load_or_cache(Path("../data/processed/X_val.npy"), val_df, "../data/raw/")
X_test = load_or_cache(Path("../data/processed/X_test.npy"), test_df, "../data/raw/")

In [9]:
X_train

array([[[-1.190e-01, -5.500e-02,  6.400e-02, ..., -2.600e-02,
         -3.900e-02, -7.900e-02],
        [-1.160e-01, -5.100e-02,  6.500e-02, ..., -3.100e-02,
         -3.400e-02, -7.400e-02],
        [-1.200e-01, -4.400e-02,  7.600e-02, ..., -2.800e-02,
         -2.900e-02, -6.900e-02],
        ...,
        [ 6.900e-02,  0.000e+00, -6.900e-02, ...,  2.400e-02,
         -4.100e-02, -5.800e-02],
        [ 8.600e-02,  4.000e-03, -8.100e-02, ...,  2.420e-01,
         -4.600e-02, -9.800e-02],
        [ 2.200e-02, -3.100e-02, -5.400e-02, ...,  1.430e-01,
         -3.500e-02, -1.200e-01]],

       [[ 4.000e-03,  1.380e-01,  1.340e-01, ...,  1.920e-01,
          8.300e-02,  8.800e-02],
        [-2.000e-02,  1.160e-01,  1.360e-01, ...,  1.560e-01,
          5.700e-02,  6.300e-02],
        [-5.300e-02,  9.200e-02,  1.450e-01, ...,  1.070e-01,
          1.300e-02,  2.200e-02],
        ...,
        [ 1.210e-01,  3.980e-01,  2.770e-01, ..., -1.065e+00,
         -4.920e-01, -1.560e-01],
        [-3.

In [10]:
features_train = extract_features(X_train)
print(features_train.shape == (17084, 48))
features_val = extract_features(X_val)
print(features_val.shape == (2146, 48))

np.isnan(features_train).any()

True
True


np.False_

In [11]:
model = make_pipeline(
    StandardScaler(),
    OneVsRestClassifier(LogisticRegression(class_weight="balanced", max_iter=1000)),
)

model.fit(features_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('onevsrestclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](5,)","[0,1,2,3,4]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,48
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [12]:
y_score = model.predict_proba(features_val)
y_score.shape

(2146, 5)

In [13]:
roc_auc_classes = np.zeros(len(mlb.classes_))
for i in range(len(mlb.classes_)):
    roc_auc_classes[i] = roc_auc_score(y_val[:, i], y_score[:, i])

roc_auc_score(y_val, y_score, average="macro")

0.7825655964003344

In [14]:
pd.Series(roc_auc_classes, index=mlb.classes_)

CD      0.815465
HYP     0.849566
MI      0.762004
NORM    0.775328
STTC    0.710465
dtype: float64

Best classification for HYP with an AUROC of 0.85 and worst classification fpr STTC with an AUROC of 0.71.